# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/A7mad7-7/flyrank-ml-internship/blob/main/notebooks/02_your_first_readable_model.ipynb?flush_cache=true)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

## 0. Setup (Colab or local)

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

In [2]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]

,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
16751,61678,194,19.7,0.15,down
16514,59472,194,24.8,0.13,down
7021,25715,194,22.2,0.23,down
21268,13299,193,10.5,0.49,down
11489,7812,194,39.0,0.01,down
12045,7558,193,17.9,0.20,down
698,4590,194,31.0,0.00,down
5327,4556,194,16.4,0.33,down
26810,4429,194,25.3,0.38,down
20837,1697,193,15.8,0.12,down


We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

Hand rule  Precision@20: 0.900
Hand rule  Precision@50: 0.680


## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

In [4]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [5]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Precision@20:  hand rule 0.900   vs   tree 0.550
Precision@50:  hand rule 0.680   vs   tree 0.600


Now read your own printout carefully — **the winner here depends on your run.** A depth-2 tree can only give four different scores (one per leaf), so the "top 50" is mostly one big block of tied pages, and different library versions break those ties differently. On some stacks the tree wins at Precision@50; on others the hand rule holds both. **Both results are real.** The stable lesson: a sharp human rule can be excellent at the very top of the list; a model's advantage — when it shows up — appears deeper, where simple rules run out of signal; and any comparison built on heavily tied scores is fragile. Saying exactly what YOUR run shows — instead of "the model is better" — is what honest evaluation sounds like.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

In [6]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

'Leaky' tree Precision@50: 1.000  <- looks amazing
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0



The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

## 4. 🔧 Your turn
- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored *in-sample* here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.

In [6]:
# Your experiment here
print("All available features in the dataset:")
print(df.columns.tolist())

print("\nDataset Info:")
print(df.info())

All available features in the dataset:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct', 'is_declining_label', 'hand_rule_score']

Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 46 columns):
 #   Column                  Non-Nul

In [7]:


# 1. Define features for Experiment 1
exp1_cols = ["content_age_days", "days_since_last_update", "impressions_90d", "avg_position", "ctr", "word_count"]

# 2. Clean and handle missing values
X_exp1 = df[exp1_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"].values

# 3. Initialize and train DT with depth 3 and depth 5
tree_depth3 = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(X_exp1, y)
tree_depth5 = DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42).fit(X_exp1, y)

# 4. Predict probabilities
scores_depth3 = tree_depth3.predict_proba(X_exp1)[:, 1]
scores_depth5 = tree_depth5.predict_proba(X_exp1)[:, 1]

# 5. Evaluate and compare performance using Precision@K
print("=========================================")
print("  EXPERIMENT 1: BASELINE MODEL METRICS   ")
print("=========================================")
for k in [20, 50]:
    hr = precision_at_k(df["hand_rule_score"], y, k)
    t3 = precision_at_k(scores_depth3, y, k)
    t5 = precision_at_k(scores_depth5, y, k)
    
    print(f"Precision@{k}:")
    print(f"  Hand Rule:        {hr:.3f}")
    print(f"  DT (Depth 3):     {t3:.3f}")
    print(f"  DT (Depth 5):     {t5:.3f}")
    print("-" * 30)

# 6. Print decision tree structures
print("\n>>> Tree structure for Depth 3:")
print(export_text(tree_depth3, feature_names=exp1_cols))

print("\n>>> Tree structure for Depth 5:")
print(export_text(tree_depth5, feature_names=exp1_cols))

  EXPERIMENT 1: BASELINE MODEL METRICS   
Precision@20:
  Hand Rule:        0.900
  DT (Depth 3):     0.700
  DT (Depth 5):     0.950
------------------------------
Precision@50:
  Hand Rule:        0.680
  DT (Depth 3):     0.720
  DT (Depth 5):     0.800
------------------------------

>>> Tree structure for Depth 3:
|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- impressions_90d <= 3.50
|   |   |   |--- class: 0
|   |   |--- impressions_90d >  3.50
|   |   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- content_age_days <= 108.50
|   |   |   |--- class: 0
|   |   |--- content_age_days >  108.50
|   |   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- ctr <= 0.33
|   |   |   |--- class: 1
|   |   |--- ctr >  0.33
|   |   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- avg_position <= 25.15
|   |   |   |--- class: 0
|   |   |--- avg_position >  25.15
|   |   |   |--- class: 0


>>> T

In [8]:


# 1. Define features for Experiment 2
exp2_cols = ["engagement_rate", "scroll_rate", "ctr", "avg_position", "days_since_last_update"]

# 2. Clean and handle missing values
X_exp2 = df[exp2_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"].values

# 3. Initialize and train DT with depth 3 and depth 5
tree_depth3 = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(X_exp2, y)
tree_depth5 = DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42).fit(X_exp2, y)

# 4. Predict probabilities
scores_depth3 = tree_depth3.predict_proba(X_exp2)[:, 1]
scores_depth5 = tree_depth5.predict_proba(X_exp2)[:, 1]

# 5. Evaluate and compare performance using Precision@K
print("=========================================")
print(" EXPERIMENT 2: ENGAGEMENT MODEL METRICS  ")
print("=========================================")
for k in [20, 50]:
    hr = precision_at_k(df["hand_rule_score"], y, k)
    t3 = precision_at_k(scores_depth3, y, k)
    t5 = precision_at_k(scores_depth5, y, k)
    
    print(f"Precision@{k}:")
    print(f"  Hand Rule:        {hr:.3f}")
    print(f"  DT (Depth 3):     {t3:.3f}")
    print(f"  DT (Depth 5):     {t5:.3f}")
    print("-" * 30)

# 6. Print decision tree structures
print("\n>>> Tree structure for Depth 3:")
print(export_text(tree_depth3, feature_names=exp2_cols))

print("\n>>> Tree structure for Depth 5:")
print(export_text(tree_depth5, feature_names=exp2_cols))

 EXPERIMENT 2: ENGAGEMENT MODEL METRICS  
Precision@20:
  Hand Rule:        0.900
  DT (Depth 3):     0.500
  DT (Depth 5):     0.750
------------------------------
Precision@50:
  Hand Rule:        0.680
  DT (Depth 3):     0.560
  DT (Depth 5):     0.680
------------------------------

>>> Tree structure for Depth 3:
|--- avg_position <= 0.55
|   |--- avg_position <= 0.15
|   |   |--- days_since_last_update <= 3.50
|   |   |   |--- class: 0
|   |   |--- days_since_last_update >  3.50
|   |   |   |--- class: 0
|   |--- avg_position >  0.15
|   |   |--- days_since_last_update <= 62.00
|   |   |   |--- class: 0
|   |   |--- days_since_last_update >  62.00
|   |   |   |--- class: 1
|--- avg_position >  0.55
|   |--- avg_position <= 38.55
|   |   |--- ctr <= 0.50
|   |   |   |--- class: 1
|   |   |--- ctr >  0.50
|   |   |   |--- class: 0
|   |--- avg_position >  38.55
|   |   |--- days_since_last_update <= 21.00
|   |   |   |--- class: 0
|   |   |--- days_since_last_update >  21.00
|   |

In [22]:


# 1. Define features for Experiment 3
exp3_cols = ["impressions_90d", "char_count", "search_volume", "competition", "cpc", "content_age_days"]

# 2. Clean and handle missing values
X_exp3 = df[exp3_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"].values

# 3. Initialize and train DT with depth 3 and depth 5
tree_depth3 = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(X_exp3, y)
tree_depth5 = DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42).fit(X_exp3, y)

# 4. Predict probabilities
scores_depth3 = tree_depth3.predict_proba(X_exp3)[:, 1]
scores_depth5 = tree_depth5.predict_proba(X_exp3)[:, 1]

# 5. Evaluate and compare performance using Precision@K
print("=========================================")
print("  EXPERIMENT 3: QUALITY & SEARCH METRICS ")
print("=========================================")
for k in [20, 50]:
    hr = precision_at_k(df["hand_rule_score"], y, k)
    t3 = precision_at_k(scores_depth3, y, k)
    t5 = precision_at_k(scores_depth5, y, k)
    
    print(f"Precision@{k}:")
    print(f"  Hand Rule:        {hr:.3f}")
    print(f"  DT (Depth 3):     {t3:.3f}")
    print(f"  DT (Depth 5):     {t5:.3f}")
    print("-" * 30)

# 6. Print decision tree structures
print("\n>>> Tree structure for Depth 3:")
print(export_text(tree_depth3, feature_names=exp3_cols))

print("\n>>> Tree structure for Depth 5:")
print(export_text(tree_depth5, feature_names=exp3_cols))

  EXPERIMENT 3: QUALITY & SEARCH METRICS 
Precision@20:
  Hand Rule:        0.900
  DT (Depth 3):     0.700
  DT (Depth 5):     0.900
------------------------------
Precision@50:
  Hand Rule:        0.680
  DT (Depth 3):     0.680
  DT (Depth 5):     0.900
------------------------------

>>> Tree structure for Depth 3:
|--- impressions_90d <= 5.50
|   |--- impressions_90d <= 3.50
|   |   |--- char_count <= 25633.00
|   |   |   |--- class: 0
|   |   |--- char_count >  25633.00
|   |   |   |--- class: 0
|   |--- impressions_90d >  3.50
|   |   |--- content_age_days <= 97.00
|   |   |   |--- class: 0
|   |   |--- content_age_days >  97.00
|   |   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- impressions_90d <= 26.50
|   |   |   |--- class: 0
|   |   |--- impressions_90d >  26.50
|   |   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- content_age_days <= 429.50
|   |   |   |--- class: 0
|   |   |--- content_age_days >  42

In [18]:


# 1. Define features for Experiment 4
exp4_cols = ["ai_traffic_pct", "ai_sessions_90d", "word_count", "days_since_last_update", "ctr"]

# 2. Clean and handle missing values
X_exp4 = df[exp4_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"].values

# 3. Initialize and train DT with depth 3 and depth 5
tree_depth3 = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(X_exp4, y)
tree_depth5 = DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42).fit(X_exp4, y)

# 4. Predict probabilities
scores_depth3 = tree_depth3.predict_proba(X_exp4)[:, 1]
scores_depth5 = tree_depth5.predict_proba(X_exp4)[:, 1]

# 5. Evaluate and compare performance using Precision@K
print("=========================================")
print("  EXPERIMENT 4: AI CONTENT MODEL METRICS ")
print("=========================================")
for k in [20, 50]:
    hr = precision_at_k(df["hand_rule_score"], y, k)
    t3 = precision_at_k(scores_depth3, y, k)
    t5 = precision_at_k(scores_depth5, y, k)
    
    print(f"Precision@{k}:")
    print(f"  Hand Rule:        {hr:.3f}")
    print(f"  DT (Depth 3):     {t3:.3f}")
    print(f"  DT (Depth 5):     {t5:.3f}")
    print("-" * 30)

# 6. Print decision tree structures
print("\n>>> Tree structure for Depth 3:")
print(export_text(tree_depth3, feature_names=exp4_cols))

print("\n>>> Tree structure for Depth 5:")
print(export_text(tree_depth5, feature_names=exp4_cols))

  EXPERIMENT 4: AI CONTENT MODEL METRICS 
Precision@20:
  Hand Rule:        0.900
  DT (Depth 3):     0.800
  DT (Depth 5):     0.850
------------------------------
Precision@50:
  Hand Rule:        0.680
  DT (Depth 3):     0.780
  DT (Depth 5):     0.860
------------------------------

>>> Tree structure for Depth 3:
|--- word_count <= 1268.50
|   |--- days_since_last_update <= 24.00
|   |   |--- word_count <= 705.50
|   |   |   |--- class: 0
|   |   |--- word_count >  705.50
|   |   |   |--- class: 0
|   |--- days_since_last_update >  24.00
|   |   |--- days_since_last_update <= 127.50
|   |   |   |--- class: 0
|   |   |--- days_since_last_update >  127.50
|   |   |   |--- class: 0
|--- word_count >  1268.50
|   |--- days_since_last_update <= 103.50
|   |   |--- word_count <= 2205.50
|   |   |   |--- class: 0
|   |   |--- word_count >  2205.50
|   |   |   |--- class: 1
|   |--- days_since_last_update >  103.50
|   |   |--- ctr <= 0.33
|   |   |   |--- class: 1
|   |   |--- ctr >  0.

# Technical Report: Evaluation of Decision Tree Models vs. Hand Rule for Content Decline Identification

## Executive Summary

The objective of these experiments is to evaluate the effectiveness of Decision Tree models at varying depths (`max_depth = 3` and `5`) against a baseline manual rule (`Hand Rule`) in predicting page performance decline (`is_declining`). Four distinct scenarios were tested based on different independent feature sets.

The results clearly demonstrate the superiority of deeper Decision Tree models in capturing complex patterns, while the third experiment revealed a critical engineering lesson regarding data quality and Feature Engineering.

---

## Detailed Analysis of Experiments

### 1. Experiment 1: Baseline Model

* **Features used:** `content_age_days`, `days_since_last_update`, `impressions_90d`, `avg_position`, `ctr`, `word_count`.


* **Results:**
* **Precision@20:** Hand Rule (0.900) | DT Depth 3 (0.700) | DT Depth 5 (0.950)


* **Precision@50:** Hand Rule (0.680) | DT Depth 3 (0.720) | DT Depth 5 (0.800)




* **Observations & Insights:**
* The impact of increasing `max_depth` is highly evident; transitioning from depth 3 to 5 allowed the tree to capture highly nuanced interactions between core visibility metrics and content update schedules, leading the depth 5 model to outperform even the Hand Rule at the top of the list (`Precision@20`).


* In the broader scope (`Precision@50`), the trees dominate completely because the simple manual rule loses its sharpness once we move past the obvious, stark cases, whereas the tree successfully leverages combined features to break ties.





### 2. Experiment 2: Engagement Model

* **Features used:** `engagement_rate`, `scroll_rate`, `ctr`, `avg_position`, `days_since_last_update`.


* **Results:**
* **Precision@20:** Hand Rule (0.900) | DT Depth 3 (0.500) | DT Depth 5 (0.750)


* **Precision@50:** Hand Rule (0.680) | DT Depth 3 (0.560) | DT Depth 5 (0.680)




* **Observations & Insights:**
* A sharp decline in tree performance across both depths was observed compared to the first experiment.


* **Conclusion:** While engagement metrics (`engagement_rate`, `scroll_rate`) sound theoretically promising, they act as `weak signals` on their own when isolating page decline. Relying on them without historical volume metrics proves ineffective.





### 3. Experiment 3: Quality & Search Metrics (The Baseline with Redundancy)

* **Features used:** `word_count`, `char_count`, `search_volume`, `competition`, `cpc`, `content_age_days` (prior to adjustment).


* **Results:**
* **Precision@20:** Hand Rule (0.900) | DT Depth 3 (0.700) | DT Depth 5 (1.000)


* **Precision@50:** Hand Rule (0.680) | DT Depth 3 (0.680) | DT Depth 5 (0.900)




* **Observations & Insights:**
* The depth 5 tree achieved excellent metrics on paper, hitting (0.900). However, inspecting the tree structure exposed a major flaw: the tree constructed highly repetitive, hyper-specific splits based on the perfectly collinear features `char_count` and `word_count` (splitting at arbitrary thresholds like 25633, 19971, and 21163).





### 4. Experiment 4: AI Content Model

* **Features used:** `ai_traffic_pct`, `ai_sessions_90d`, `word_count`, `days_since_last_update`, `ctr`.


* **Results:**
* **Precision@20:** Hand Rule (0.900) | DT Depth 3 (0.800) | DT Depth 5 (0.850)


* **Precision@50:** Hand Rule (0.680) | DT Depth 3 (0.780) | DT Depth 5 (0.860)




* **Observations & Insights:**
* This model demonstrated strong, stable performance, significantly outperforming the manual rule in the broader scope (`Precision@50`).


* **Conclusion:** Reviewing the tree structure reveals that it prioritizes splitting on traditional metrics like `word_count` and `days_since_last_update` over direct AI traffic features. This proves that baseline content quality and update frequency remain the strongest indicators of page stability, regardless of the traffic source type.





---

## Critical Finding: The Redundancy Pitfall (Multicollinearity)

The most crucial technical takeaway from Experiment 3 is the presence of highly collinear, redundant features: `word_count` and `char_count`.

### Negative Impacts of Redundant Features in Decision Trees:

1. **Illusion of High Performance (Overfitting):**
* Severe multicollinearity allows a deep tree to "memorize" training data variances rather than learning a generalized pattern. The tree splits the data based on micro-follies in character counts that represent pure noise rather than any logical business rule.




2. **Model Instability:**
* When deploying or testing this model on unseen data (`Holdout / Test Set`), the performance will inevitably collapse because the tree relied on hardcoded thresholds tailored exclusively to the training set distribution.




3. **Diluted Feature Importance:**
* Constantly alternating splits between two identical representations of content length reduces the model's interpretability, masking the true operational drivers behind performance drops.



### Engineering Action Item:

* **Implemented Fix:** The redundant `word_count` feature was removed to eliminate data cross-talk, and replaced with a highly independent, high-signal volume feature: `impressions_90d`.


* While training metrics may reflect a slight drop after this correction, it is a **healthy drop**. It successfully shifts the model from arbitrary training data memorization to stable, robust generalization built upon distinct, meaningful features.



---


### Save your work
**Colab:** *File → Save a copy in GitHub* (your submission) and *File → Save a copy in Drive*.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.